# Entropy & Loss Analysis: HML Benchmarks

**Per-token entropy and cross-entropy loss across:**
- 15 models (requested DeepSeek/Qwen/Llama/GPT-OSS/Gemma/Phi suite)
- 3 datasets: HumanEval, MBPP, LiveCodeBench
- 2 modes: Regular (CoT), Chain_Code

**Key Questions:**
1. How does entropy/loss differ across benchmark difficulty?
2. How do models compare on the same dataset?
3. What layer-wise patterns emerge?
4. How does chain_code mode differ from regular?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

BASE_DIR = Path("../../results/hml")
REQUESTED_MODELS = [
    'Qwen--Qwen3-0.6B',
    'Qwen--Qwen3-4B',
    'Qwen--Qwen3-8B',
    'Qwen--Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b',
    'google--gemma-4-E2B-it',
    'google--gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning',
]
MODEL_DIR_BY_REQUEST = {
    'Qwen--Qwen3-0.6B': 'Qwen3-0.6B',
    'Qwen--Qwen3-4B': 'Qwen3-4B',
    'Qwen--Qwen3-8B': 'Qwen3-8B',
    'Qwen--Qwen3-14B': 'Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B': 'DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B': 'DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B': 'DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B': 'DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B': 'DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct': 'Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b': 'gpt-oss-20b',
    'google--gemma-4-E2B-it': 'gemma-4-E2B-it',
    'google--gemma-4-E4B-it': 'gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it': 'gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning': 'Phi-4-mini-reasoning',
}
MODELS = [MODEL_DIR_BY_REQUEST[m] for m in REQUESTED_MODELS]
MODEL_DISPLAY = {MODEL_DIR_BY_REQUEST[m]: m for m in REQUESTED_MODELS}
MODEL_SHORT = {
    'Qwen3-0.6B': 'Q3-0.6B',
    'Qwen3-4B': 'Q3-4B',
    'Qwen3-8B': 'Q3-8B',
    'Qwen3-14B': 'Q3-14B',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'DS-DQ1.5B',
    'DeepSeek-R1-Distill-Qwen-7B': 'DS-DQ7B',
    'DeepSeek-R1-Distill-Llama-8B': 'DS-DL8B',
    'DeepSeek-R1-Distill-Qwen-14B': 'DS-DQ14B',
    'DeepSeek-R1-0528-Qwen3-8B': 'DS-0528-Q8B',
    'Llama-3.2-3B-Instruct': 'Llama3.2-3B',
    'gpt-oss-20b': 'gpt-oss-20b',
    'gemma-4-E2B-it': 'Gemma4-E2B',
    'gemma-4-E4B-it': 'Gemma4-E4B',
    'gemma-4-26B-A4B-it': 'Gemma4-26B',
    'Phi-4-mini-reasoning': 'Phi4-mini',
}
DATASETS = ["humaneval", "mbpp", "livecodebench"]
MODES = {"CoT": "entropy_loss_results", "Chain_Code": "entropy_loss_results_chain_code"}

palette = sns.color_palette('tab20', n_colors=max(12, len(MODELS)))
MODEL_COLORS = {model: palette[i] for i, model in enumerate(MODELS)}
DS_COLORS = {"humaneval": "tab:green", "mbpp": "tab:purple", "livecodebench": "tab:red"}


def model_label(model, short=False):
    if short:
        return MODEL_SHORT.get(model, model)
    return MODEL_DISPLAY.get(model, model)

## 1. Data Loading

In [ ]:
def load_entropy_data(dataset, model, mode_dir):
    """Load per_token.npz for entropy/loss data."""
    path = BASE_DIR / dataset / mode_dir / model / dataset / "per_token.npz"
    if not path.exists():
        return None
    data = np.load(path)
    
    entropy_keys = sorted([k for k in data.keys() if k.startswith('entropy_layer_')],
                          key=lambda k: int(k.split('_')[-1]))
    loss_keys = sorted([k for k in data.keys() if k.startswith('loss_layer_')],
                       key=lambda k: int(k.split('_')[-1]))
    
    num_layers = len(entropy_keys)
    avg_entropy = np.array([np.nanmean(data[k]) for k in entropy_keys])
    avg_loss = np.array([np.nanmean(data[k]) for k in loss_keys])
    std_entropy = np.array([np.nanstd(data[k]) for k in entropy_keys])
    std_loss = np.array([np.nanstd(data[k]) for k in loss_keys])
    
    return {
        'avg_entropy': avg_entropy, 'avg_loss': avg_loss,
        'std_entropy': std_entropy, 'std_loss': std_loss,
        'num_layers': num_layers,
        'per_layer_entropy': [data[k] for k in entropy_keys],
        'per_layer_loss': [data[k] for k in loss_keys],
    }

# Load all data
all_data = {}  # {mode: {model: {dataset: data}}}
for mode_name, mode_dir in MODES.items():
    all_data[mode_name] = {}
    for model in MODELS:
        all_data[mode_name][model] = {}
        print(f"=== {mode_name} | {model} ===")
        for ds in DATASETS:
            data = load_entropy_data(ds, model, mode_dir)
            if data:
                all_data[mode_name][model][ds] = data
                print(f"  {ds}: {data['num_layers']} layers, final entropy={data['avg_entropy'][-1]:.4f}, loss={data['avg_loss'][-1]:.4f}")
            else:
                print(f"  {ds}: NOT FOUND")
        print()

## 2. Overview: Final-Layer Metrics

In [ ]:
summary_data = []
for mode_name in MODES:
    for model in MODELS:
        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data is None:
                continue
            summary_data.append({
                'Mode': mode_name, 'Model': model, 'Dataset': ds,
                'Final Entropy': data['avg_entropy'][-1],
                'Final Loss': data['avg_loss'][-1],
                'Mean Entropy': data['avg_entropy'].mean(),
                'Mean Loss': data['avg_loss'].mean(),
            })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False, float_format='%.4f'))

## 3. Same Model, Different Datasets

How does each model's behavior differ across benchmarks?

In [ ]:
for mode_name in MODES:
    n_cols = min(4, len(MODELS))
    n_rows = int(np.ceil(len(MODELS) / n_cols))

    fig_ent, axes_ent = plt.subplots(n_rows, n_cols, figsize=(5.6 * n_cols, 4.5 * n_rows), squeeze=False)
    fig_loss, axes_loss = plt.subplots(n_rows, n_cols, figsize=(5.6 * n_cols, 4.5 * n_rows), squeeze=False)

    for idx, model in enumerate(MODELS):
        r, c = divmod(idx, n_cols)
        ax_ent = axes_ent[r, c]
        ax_loss = axes_loss[r, c]

        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data is None:
                continue
            x = np.arange(1, data['num_layers'] + 1)
            color = DS_COLORS[ds]
            ax_ent.plot(x, data['avg_entropy'], color=color, linewidth=2, label=ds)
            ax_ent.fill_between(
                x,
                data['avg_entropy'] - data['std_entropy'],
                data['avg_entropy'] + data['std_entropy'],
                alpha=0.15,
                color=color,
            )
            ax_loss.plot(x, data['avg_loss'], color=color, linewidth=2, label=ds)
            ax_loss.fill_between(
                x,
                data['avg_loss'] - data['std_loss'],
                data['avg_loss'] + data['std_loss'],
                alpha=0.15,
                color=color,
            )

        short_name = model_label(model, short=True)
        ax_ent.set_xlabel('Layer')
        ax_ent.set_ylabel('Entropy')
        ax_ent.set_title(short_name, fontsize=11)
        ax_ent.legend(fontsize=8)

        ax_loss.set_xlabel('Layer')
        ax_loss.set_ylabel('Loss')
        ax_loss.set_title(short_name, fontsize=11)
        ax_loss.legend(fontsize=8)

    total_slots = n_rows * n_cols
    for empty_idx in range(len(MODELS), total_slots):
        r, c = divmod(empty_idx, n_cols)
        axes_ent[r, c].axis('off')
        axes_loss[r, c].axis('off')

    fig_ent.suptitle(f'Layer-wise Entropy by Model ({mode_name})', fontsize=15, y=1.01)
    fig_ent.tight_layout()
    plt.show()

    fig_loss.suptitle(f'Layer-wise Loss by Model ({mode_name})', fontsize=15, y=1.01)
    fig_loss.tight_layout()
    plt.show()

## 4. Different Models, Same Dataset

In [ ]:
for mode_name in MODES:
    # Increased columns to 3 and width to 24 to accommodate the new plots
    fig, axes = plt.subplots(len(DATASETS), 3, figsize=(24, 6*len(DATASETS)), squeeze=False)
    
    for idx, ds in enumerate(DATASETS):
        # Unpack 3 axes per row now
        ax_ent, ax_loss_all, ax_loss_filtered = axes[idx]
        
        for model in MODELS:
            data = all_data[mode_name][model].get(ds)
            if data is None:
                continue
            x = np.linspace(0, 1, data['num_layers'])
            color = MODEL_COLORS[model]
            label = model_label(model, short=True)
            
            # 1. Plot Entropy (shading removed)
            ax_ent.plot(x, data['avg_entropy'], color=color, linewidth=2, label=label)
            
            # 2. Plot Loss - All Models (shading removed)
            ax_loss_all.plot(x, data['avg_loss'], color=color, linewidth=2, label=label)
            
            # 3. Plot Loss - Filtered
            # The outlier spikes above 1500, so 1000 is a safe threshold to exclude it
            if np.max(data['avg_loss']) < 1000:
                ax_loss_filtered.plot(x, data['avg_loss'], color=color, linewidth=2, label=label)

        # Formatting Entropy
        ax_ent.set_xlabel('Normalized Layer')
        ax_ent.set_ylabel('Entropy')
        ax_ent.set_title(f'{ds} - Entropy', fontsize=13)
        
        # Formatting Loss (All)
        ax_loss_all.set_xlabel('Normalized Layer')
        ax_loss_all.set_ylabel('Loss')
        ax_loss_all.set_title(f'{ds} - Loss (All)', fontsize=13)
        
        # Formatting Loss (Filtered)
        ax_loss_filtered.set_xlabel('Normalized Layer')
        ax_loss_filtered.set_ylabel('Loss')
        ax_loss_filtered.set_title(f'{ds} - Loss (Filtered Zoom)', fontsize=13)

    # Legend handling
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        # Adjusted bbox slightly to fit the new 3-column width
        fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), title='Model')
        
    fig.suptitle(f'Cross-Model Comparison ({mode_name})', fontsize=15, y=1.02)
    fig.tight_layout(rect=[0, 0, 0.88, 1]) 
    plt.show()

## 5. CoT vs Chain_Code Comparison

In [ ]:
# Bar chart: final-layer entropy/loss deltas (Chain_Code - CoT)
fig, axes = plt.subplots(2, 1, figsize=(18, 12)) 

bar_data = []
for model in MODELS:
    for ds in DATASETS:
        cot = all_data['CoT'][model].get(ds)
        cc = all_data['Chain_Code'][model].get(ds)
        if cot and cc:
            bar_data.append({
                'Model': model, 'Dataset': ds,
                'Delta Entropy': cc['avg_entropy'][-1] - cot['avg_entropy'][-1],
                'Delta Loss': cc['avg_loss'][-1] - cot['avg_loss'][-1],
            })

bar_df = pd.DataFrame(bar_data)
if not bar_df.empty:
    model_order = {m: i for i, m in enumerate(MODELS)}
    dataset_order = {d: i for i, d in enumerate(DATASETS)}
    bar_df['ModelOrder'] = bar_df['Model'].map(model_order)
    bar_df['DatasetOrder'] = bar_df['Dataset'].map(dataset_order)
    
    # Reset index after sorting so our x array matches the dataframe index perfectly
    bar_df = bar_df.sort_values(['ModelOrder', 'DatasetOrder']).reset_index(drop=True)

    x = np.arange(len(bar_df))
    colors = [MODEL_COLORS[m] for m in bar_df['Model']]
    
    # Calculate a single centered tick position for each model
    tick_positions = []
    tick_labels = []
    for model in bar_df['Model'].unique():
        # Get the indices of the bars for this specific model
        model_indices = bar_df[bar_df['Model'] == model].index
        
        # Use np.mean() to calculate the center position safely
        tick_positions.append(np.mean(model_indices)) 
        tick_labels.append(MODEL_SHORT.get(model, model))
    
    # Top Row: Entropy
    axes[0].bar(x, bar_df['Delta Entropy'], color=colors)
    axes[0].set_xticks(tick_positions)
    axes[0].set_xticklabels(tick_labels, fontsize=10, rotation=35, ha='right')
    axes[0].set_ylabel('Delta Entropy (CC - CoT)')
    axes[0].set_title('Final Entropy Delta', fontsize=13)
    axes[0].axhline(0, color='black', linewidth=0.5)
    
    # Bottom Row: Loss
    axes[1].bar(x, bar_df['Delta Loss'], color=colors)
    axes[1].set_xticks(tick_positions)
    axes[1].set_xticklabels(tick_labels, fontsize=10, rotation=35, ha='right')
    axes[1].set_ylabel('Delta Loss (CC - CoT)')
    axes[1].set_title('Final Loss Delta', fontsize=13)
    axes[1].axhline(0, color='black', linewidth=0.5)

fig.tight_layout()
plt.show()

In [ ]:
# Layer-wise comparison: CoT vs Chain_Code for each model+dataset
for model in MODELS:
    fig, axes = plt.subplots(len(DATASETS), 2, figsize=(18, 5*len(DATASETS)), squeeze=False)
    for row, ds in enumerate(DATASETS):
        cot = all_data['CoT'][model].get(ds)
        cc = all_data['Chain_Code'][model].get(ds)
        ax_ent, ax_loss = axes[row]
        for mode_name, data, style in [('CoT', cot, '-'), ('Chain_Code', cc, '--')]:
            if data is None:
                continue
            x = np.arange(1, data['num_layers']+1)
            ax_ent.plot(x, data['avg_entropy'], style, linewidth=2, label=mode_name)
            ax_loss.plot(x, data['avg_loss'], style, linewidth=2, label=mode_name)
        ax_ent.set_xlabel('Layer'); ax_ent.set_ylabel('Entropy')
        ax_ent.set_title(f'{ds} - Entropy', fontsize=12); ax_ent.legend()
        ax_loss.set_xlabel('Layer'); ax_loss.set_ylabel('Loss')
        ax_loss.set_title(f'{ds} - Loss', fontsize=12); ax_loss.legend()
    fig.suptitle(f'{model} - CoT vs Chain_Code', fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()

## 6. Per-Token Distribution (Final Layer)

In [ ]:
for model in MODELS:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(7*len(DATASETS), 5), squeeze=False)
    for col, ds in enumerate(DATASETS):
        ax = axes[0, col]
        for mode_name, mode_dir in MODES.items():
            data = all_data[mode_name][model].get(ds)
            if data is None:
                continue
            final_entropy = data['per_layer_entropy'][-1]
            final_entropy = final_entropy[np.isfinite(final_entropy)]
            ax.hist(final_entropy, bins=50, alpha=0.5, label=mode_name, density=True)
        ax.set_xlabel('Entropy'); ax.set_ylabel('Density')
        ax.set_title(f'{ds}', fontsize=12)
        ax.legend()
    fig.suptitle(f'{model} - Final Layer Entropy Distribution', fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()

## 7. Summary

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS: ENTROPY & LOSS ANALYSIS (HML)")
print("="*80)

for model in MODELS:
    print(f"\n{model}:")
    for mode_name in MODES:
        print(f"  {mode_name}:")
        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data:
                print(f"    {ds:15s}  final_entropy={data['avg_entropy'][-1]:.4f}  final_loss={data['avg_loss'][-1]:.4f}")
            else:
                print(f"    {ds:15s}  NO DATA")